# 02 — Data Prep & Feature Engineering

Daily -> weekly aggregation for the top-10 container ports. Features: lags 1–8w, rolling mean/std (4/8/12w), calendar (week-of-year sin/cos, month, Ramadan, Lebaran).

In [1]:
import sys; sys.path.insert(0, '.')
import pandas as pd
from src.data import build_weekly_top, top_ports
from src.features import build_features

weekly = build_weekly_top(n_ports=10)
print(f'{len(weekly)} port-weeks, {weekly.portname.nunique()} ports, {weekly.year_week.min()}..{weekly.year_week.max()}')
print(weekly.portname.unique().tolist())

3973 port-weeks, 10 ports, 201902..202634
['Belawan', 'Bitung', 'Donggala', 'Gresik', 'Kendari', 'Makassar', 'Manokwari Road', 'Surabaya', 'Tanjung Priok', 'Teluk Bayur']


Full weeks only (7 daily rows). Train/test split for walk-forward: test starts 2025-W01 (~86 weeks).

In [2]:
w = weekly.copy()
from src.features import _week_start
w['week_mon'] = _week_start(w.year_week.astype(int)).dt.strftime('%Y-%m-%d')
print('weeks per port:'); print(w.groupby('portname').size().to_string())
print('\nsample weeks:'); print(w[['year_week','week_mon']].drop_duplicates().head(3).to_string(index=False))

weeks per port:
portname
Belawan           398
Bitung            398
Donggala          398
Gresik            398
Kendari           391
Makassar          398
Manokwari Road    398
Surabaya          398
Tanjung Priok     398
Teluk Bayur       398

sample weeks:
 year_week   week_mon
    201902 2019-01-07
    201903 2019-01-14
    201904 2019-01-21


In [3]:
feats = build_features(w[w.portname=='Tanjung Priok'], target='portcalls_container')
print(feats.shape)
print(feats.columns.tolist())

(386, 27)
['portname', 'year_week', 'portcalls_container', 'import_container', 'export_container', 'week_start', 'week_mon', 'lag_1', 'lag_2', 'lag_3', 'lag_4', 'lag_5', 'lag_6', 'lag_7', 'lag_8', 'roll_mean_4', 'roll_std_4', 'roll_mean_8', 'roll_std_8', 'roll_mean_12', 'roll_std_12', 'week_of_year', 'month', 'woy_sin', 'woy_cos', 'ramadan', 'lebaran']


Feature matrix preview: lag_1..8, rolling 4/8/12 mean+std, calendar flags. First 12 weeks dropped (warm-up). NaN-free at fit time.

In [4]:
print(feats[['year_week','portcalls_container','lag_1','lag_8','roll_mean_12','ramadan','lebaran']].head(8).to_string(index=False))

 year_week  portcalls_container  lag_1  lag_8  roll_mean_12  ramadan  lebaran
    201914                   79   80.0   84.0     80.333333        0        0
    201915                   85   79.0   75.0     80.000000        0        0
    201916                   78   85.0   72.0     81.166667        0        0
    201917                   78   78.0   75.0     80.916667        0        0
    201918                   73   78.0   95.0     80.500000        0        0
    201919                   69   73.0   79.0     79.583333        1        0
    201920                   82   69.0   86.0     79.083333        1        0
    201921                   78   82.0   80.0     79.916667        1        0
